## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-22-16-46-00 +0000,wsj,A New Figure Emerges in the Nvidia Chip-Smuggl...,https://www.wsj.com/articles/a-new-figure-emer...
1,2026-11-22-16-45-41 +0000,nyt,FAA Says All Delta Flights Delayed as Automate...,https://www.nytimes.com/2026/11/22/us/delta-fl...
2,2026-11-22-16-45-30 +0000,wapo,Mark Carney brings Trump impersonator onstage ...,https://www.washingtonpost.com/world/2026/11/2...
3,2026-11-22-16-43-00 +0000,wsj,Opinion | Iran Doubles Down on Its Nuclear Amb...,https://www.wsj.com/opinion/iran-doubles-down-...
4,2026-11-22-16-42-00 +0000,wsj,Opinion | Welcome to the Stephen Miller Presid...,https://www.wsj.com/opinion/welcome-to-the-ste...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2481/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
18,trump,19
153,california,8
225,fire,7
235,he,7
244,dead,7


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
23,2026-11-22-15-27-09 +0000,nypost,"Iran taunts Trump, shows off underground ‘miss...",https://nypost.com/2026/11/22/us-news/iran-sho...,53
64,2026-11-22-11-00-00 +0000,wapo,Trump keeps losing in court. Here’s how he get...,https://www.washingtonpost.com/politics/2026/1...,44
98,2026-11-22-06-43-17 +0000,nypost,Mike Waltz warns Russia faces ‘unsustainable’ ...,https://nypost.com/2026/11/22/us-news/russia-f...,37
102,2026-11-22-05-56-44 +0000,nypost,Trump says Sen. John Cornyn is doing a ‘tremen...,https://nypost.com/2026/11/22/us-news/trump-sa...,34
153,2026-11-21-19-45-46 +0000,nypost,Pelosi breaks vow to never utter ‘curse word’ ...,https://nypost.com/2026/11/21/us-news/pelosi-d...,34


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
23,53,2026-11-22-15-27-09 +0000,nypost,"Iran taunts Trump, shows off underground ‘miss...",https://nypost.com/2026/11/22/us-news/iran-sho...
36,33,2026-11-22-13-17-40 +0000,cbc,Fire at Indonesian candy factory leaves 3 dead...,https://www.cbc.ca/news/world/indonesian-candy...
77,31,2026-11-22-10-00-00 +0000,wsj,Richer and more adventurous than their predece...,https://www.wsj.com/personal-finance/retiremen...
89,28,2026-11-22-08-43-29 +0000,nypost,Ex-White House press secretary Katie Zacharia ...,https://nypost.com/2026/11/22/us-news/ex-white...
126,28,2026-11-21-23-50-12 +0000,chitri,Nations agree to extend talks on AI killer rob...,https://www.chicagotribune.com/2026/11/21/nati...
173,28,2026-11-21-17-38-58 +0000,nypost,Russian cargo ship bursts into flames in Balti...,https://nypost.com/2026/11/21/world-news/russi...
104,26,2026-11-22-04-05-27 +0000,nypost,ICE officers arrest Alaska Airlines passengers...,https://nypost.com/2026/11/21/us-news/masked-i...
64,25,2026-11-22-11-00-00 +0000,wapo,Trump keeps losing in court. Here’s how he get...,https://www.washingtonpost.com/politics/2026/1...
118,25,2026-11-22-01-25-03 +0000,bbc,Mexican drug lord on FBI's 10 most wanted list...,https://www.bbc.co.uk/news/articles/ckz749ey0r...
31,23,2026-11-22-14-10-03 +0000,nypost,Illegal migrant accused of killing grandmother...,https://nypost.com/2026/11/22/us-news/illegal-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
